In [28]:
import pandas as pd
import numpy as np

df_all = pd.read_csv(
    "../data/processed/multi_assets_supervised.csv",
    index_col=0,
    parse_dates=True
)

df_all.head()

,Close,High,Low,Open,Volume,Return,Target_Return,Target_Class,Ticker
Date,,,,,,,,,
2015-01-05,23.532721,24.064284,23.346674,23.984549,257142000,-0.028172,0.000094,1,AAPL
2015-01-06,23.534933,23.794069,23.173912,23.596948,263188400,0.000094,0.014022,1,AAPL
2015-01-07,23.864941,23.964608,23.632381,23.743124,160423600,0.014022,0.038423,1,AAPL
2015-01-08,24.781898,24.839485,24.075362,24.192751,237458000,0.038423,0.001072,1,AAPL
2015-01-09,24.808472,25.083112,24.409799,24.954651,214798000,0.001072,-0.024641,0,AAPL


In [29]:
threshold = 0.002  # 0.2%

df_all["Target_Class"] = np.where(
    df_all.groupby("Ticker")["Return"].shift(-1) > threshold, 1,
    np.where(df_all.groupby("Ticker")["Return"].shift(-1) < -threshold, 0, np.nan)
)

In [30]:
df_all = df_all.dropna(subset=["Target_Class"])

In [31]:
df_all = df_all.sort_values(["Ticker", "Date"])


# RETURN LAGS
df_all["Return_lag1"] = df_all.groupby("Ticker")["Return"].shift(1)
df_all["Return_lag2"] = df_all.groupby("Ticker")["Return"].shift(2)
df_all["Return_lag3"] = df_all.groupby("Ticker")["Return"].shift(3)


# MOMENTUM
df_all["Momentum_5"] = df_all.groupby("Ticker")["Close"].transform(lambda x: x / x.shift(5) - 1)
df_all["Momentum_10"] = df_all.groupby("Ticker")["Close"].transform(lambda x: x / x.shift(10) - 1)


# MOVING AVERAGE
df_all["MA_5"] = df_all.groupby("Ticker")["Close"].transform(lambda x: x.rolling(5).mean())
df_all["MA_20"] = df_all.groupby("Ticker")["Close"].transform(lambda x: x.rolling(20).mean())

df_all["MA_ratio"] = df_all["MA_5"] / df_all["MA_20"]


# VOLATILITÀ
df_all["Volatility_10"] = df_all.groupby("Ticker")["Return"].transform(lambda x: x.rolling(10).std())
df_all["Volatility_30"] = df_all.groupby("Ticker")["Return"].transform(lambda x: x.rolling(30).std())

df_all["Momentum_vol_adjusted"] = df_all["Momentum_10"] / df_all["Volatility_10"]

df_all["Price_vs_MA20"] = df_all["Close"] / df_all["MA_20"]

delta = df_all.groupby("Ticker")["Close"].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.groupby(df_all["Ticker"]).transform(lambda x: x.rolling(14).mean())
avg_loss = loss.groupby(df_all["Ticker"]).transform(lambda x: x.rolling(14).mean())

rs = avg_gain / avg_loss

df_all["RSI"] = 100 - (100 / (1 + rs))

In [32]:
df_all = df_all.dropna()

In [33]:
df_all["Target_Class"].value_counts(normalize=True)

Target_Class
1.0    0.538481
0.0    0.461519
Name: proportion, dtype: float64

In [34]:
df_all.to_csv("../data/processed/multi_assets_threshold_002.csv")

In [35]:
threshold = 0.001  # 0.1%

df_all["Target_Class"] = np.where(
    df_all.groupby("Ticker")["Return"].shift(-1) > threshold, 1,
    np.where(df_all.groupby("Ticker")["Return"].shift(-1) < -threshold, 0, np.nan)
)

In [36]:
df_all = df_all.dropna(subset=["Target_Class"])

In [38]:
df_all = df_all.sort_values(["Ticker", "Date"])


# RETURN LAGS
df_all["Return_lag1"] = df_all.groupby("Ticker")["Return"].shift(1)
df_all["Return_lag2"] = df_all.groupby("Ticker")["Return"].shift(2)
df_all["Return_lag3"] = df_all.groupby("Ticker")["Return"].shift(3)


# MOMENTUM
df_all["Momentum_5"] = df_all.groupby("Ticker")["Close"].transform(lambda x: x / x.shift(5) - 1)
df_all["Momentum_10"] = df_all.groupby("Ticker")["Close"].transform(lambda x: x / x.shift(10) - 1)


# MOVING AVERAGE
df_all["MA_5"] = df_all.groupby("Ticker")["Close"].transform(lambda x: x.rolling(5).mean())
df_all["MA_20"] = df_all.groupby("Ticker")["Close"].transform(lambda x: x.rolling(20).mean())

df_all["MA_ratio"] = df_all["MA_5"] / df_all["MA_20"]


# VOLATILITÀ
df_all["Volatility_10"] = df_all.groupby("Ticker")["Return"].transform(lambda x: x.rolling(10).std())
df_all["Volatility_30"] = df_all.groupby("Ticker")["Return"].transform(lambda x: x.rolling(30).std())

df_all["Momentum_vol_adjusted"] = df_all["Momentum_10"] / df_all["Volatility_10"]

df_all["Price_vs_MA20"] = df_all["Close"] / df_all["MA_20"]

delta = df_all.groupby("Ticker")["Close"].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.groupby(df_all["Ticker"]).transform(lambda x: x.rolling(14).mean())
avg_loss = loss.groupby(df_all["Ticker"]).transform(lambda x: x.rolling(14).mean())

rs = avg_gain / avg_loss

df_all["RSI"] = 100 - (100 / (1 + rs))

In [39]:
df_all = df_all.dropna()

In [40]:
df_all["Target_Class"].value_counts(normalize=True)

Target_Class
1.0    0.534505
0.0    0.465495
Name: proportion, dtype: float64

In [41]:
df_all.to_csv("../data/processed/multi_assets_threshold_001.csv")